In [1]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

print("All libraries imported successfully.")



All libraries imported successfully.


In [2]:
file_path = "Spain_Trade_Panel_2000_2025_RTA.xlsx"

df = pd.read_excel(file_path, sheet_name="Spain_Trade")

df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("Years:", df["Year"].min(), "-", df["Year"].max())
print("Unique trading partners:", df["Trade Partner"].nunique())

display(df.head())
display(df.dtypes.to_frame("Data Type"))
display(df.isnull().sum().to_frame("Missing Values"))


Dataset shape: (5886, 15)
Years: 2000 - 2025
Unique trading partners: 235


,Year,Country,Trade Partner,Export,Import,Bilateral Trade,GDP country A,GDP TP,Distance,contig,comcol,comlang_ethno,comlang_off,colony,RTA
0,2000,Spain,Afghanistan,2.744438e+05,2.157179e+05,4.901616e+05,5.981029e+11,3.521418e+09,6290.4930,0.0,0.0,0.0,0.0,0.0,0
1,2000,Spain,Albania,1.403186e+07,1.514237e+06,1.554610e+07,5.981029e+11,3.584570e+09,1977.0610,0.0,0.0,0.0,0.0,0.0,0
2,2000,Spain,Algeria,5.046210e+08,2.847340e+09,3.351961e+09,5.981029e+11,5.479040e+10,706.8757,0.0,0.0,0.0,0.0,0.0,1
3,2000,Spain,Andorra,5.331061e+08,2.187875e+07,5.549849e+08,5.981029e+11,1.432606e+09,492.9274,0.0,0.0,1.0,1.0,0.0,1
4,2000,Spain,Angola,6.567004e+07,1.835103e+08,2.491803e+08,5.981029e+11,9.129595e+09,5752.2520,0.0,0.0,0.0,0.0,0.0,0


,Data Type
Year,int64
Country,str
Trade Partner,str
Export,float64
Import,float64
Bilateral Trade,float64
GDP country A,float64
GDP TP,float64
Distance,float64
contig,float64


,Missing Values
Year,0
Country,0
Trade Partner,0
Export,0
Import,0
Bilateral Trade,0
GDP country A,0
GDP TP,703
Distance,323
contig,323


In [3]:
model_vars = [
    "Year",
    "Trade Partner",
    "Bilateral Trade",
    "GDP country A",
    "GDP TP",
    "Distance",
    "contig",
    "comlang_off",
    "colony",
    "RTA"
]

missing_cols = [
    "GDP TP", "Distance", "contig",
    "comlang_off", "colony"
]

print("Missing-value overlap:")
display(
    df[missing_cols]
    .isna()
    .value_counts()
    .rename("Observations")
    .to_frame()
)

print(
    "\nPartners with missing geographical data:",
    df.loc[df["Distance"].isna(), "Trade Partner"].nunique()
)

print(
    "Partners with missing partner GDP:",
    df.loc[df["GDP TP"].isna(), "Trade Partner"].nunique()
)

print("\nZero bilateral trade:", (df["Bilateral Trade"] == 0).sum())
print("Negative bilateral trade:", (df["Bilateral Trade"] < 0).sum())


Missing-value overlap:


Observations
GDP TP Distance contig comlang_off colony              
False  False    False  False       False           5092
True   False    False  False       False            471
       True     True   True        True             232
False  True     True   True        True              91


Partners with missing geographical data: 17
Partners with missing partner GDP: 52

Zero bilateral trade: 0
Negative bilateral trade: 0


In [5]:

model_vars = [
    "Year", "Trade Partner", "Bilateral Trade",
    "GDP country A", "GDP TP", "Distance",
    "contig", "comlang_off", "colony", "RTA"
]

model_data = df[model_vars].dropna().copy()

model_data = model_data[
    (model_data["GDP country A"] > 0) &
    (model_data["GDP TP"] > 0) &
    (model_data["Distance"] > 0) &
    (model_data["Bilateral Trade"] >= 0)
].copy()

ols_data = model_data[
    model_data["Bilateral Trade"] > 0
].copy()

ols_data["ln_trade"] = np.log(ols_data["Bilateral Trade"])
ols_data["ln_gdp_spain"] = np.log(ols_data["GDP country A"])
ols_data["ln_gdp_partner"] = np.log(ols_data["GDP TP"])
ols_data["ln_distance"] = np.log(ols_data["Distance"])

print("Original observations:", len(df))
print("Complete observations:", len(model_data))
print("OLS observations:", len(ols_data))
print("Zero-trade observations in PPML sample:",
      (model_data["Bilateral Trade"] == 0).sum())

print("OLS year range:",
      ols_data["Year"].min(), "-", ols_data["Year"].max())


Original observations: 5886
Complete observations: 5092
OLS observations: 5092
Zero-trade observations in PPML sample: 0
OLS year range: 2000 - 2025


In [7]:

ols_formula = """
ln_trade ~ ln_gdp_spain
         + ln_gdp_partner
         + ln_distance
         + contig
         + comlang_off
         + colony
         + RTA
"""

ols_model = smf.ols(
    formula=ols_formula,
    data=ols_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": ols_data["Trade Partner"]}
)

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               ln_trade   R-squared:                       0.847
Model:                            OLS   Adj. R-squared:                  0.846
Method:                 Least Squares   F-statistic:                     317.4
Date:                Mon, 12 Oct 2026   Prob (F-statistic):          5.94e-104
Time:                        18:04:03   Log-Likelihood:                -8413.5
No. Observations:                5092   AIC:                         1.684e+04
Df Residuals:                    5084   BIC:                         1.690e+04
Df Model:                           7                                         
Covariance Type:              cluster                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          8.1506      2.550      3.

In [14]:
ppml_data = model_data.copy()

ppml_data["ln_gdp_spain"] = np.log(ppml_data["GDP country A"])
ppml_data["ln_gdp_partner"] = np.log(ppml_data["GDP TP"])
ppml_data["ln_distance"] = np.log(ppml_data["Distance"])
print("PPML observations:", len(ppml_data))
print("Dependent variable: Bilateral Trade")
print("Minimum bilateral trade:", ppml_data["Bilateral Trade"].min())
print("Maximum bilateral trade:", ppml_data["Bilateral Trade"].max())

PPML observations: 5092
Dependent variable: Bilateral Trade
Minimum bilateral trade: 63.346
Maximum bilateral trade: 106180055618.265


In [10]:

ppml_formula = """
Q('Bilateral Trade') ~ ln_gdp_spain
                     + ln_gdp_partner
                     + ln_distance
                     + contig
                     + comlang_off
                     + colony
                     + RTA
"""

ppml_model = smf.glm(
    formula=ppml_formula,
    data=ppml_data,
    family=sm.families.Poisson()
).fit(
    cov_type="cluster",
    cov_kwds={"groups": ppml_data["Trade Partner"]}
)

print(ppml_model.summary())


                  Generalized Linear Model Regression Results                   
Dep. Variable:     Q('Bilateral Trade')   No. Observations:                 5092
Model:                              GLM   Df Residuals:                     5084
Model Family:                   Poisson   Df Model:                            7
Link Function:                      Log   Scale:                          1.0000
Method:                            IRLS   Log-Likelihood:            -1.4958e+12
Date:                  Mon, 12 Oct 2026   Deviance:                   2.9915e+12
Time:                          18:12:19   Pearson chi2:                 3.40e+12
No. Iterations:                      27   Pseudo R-squ. (CS):              1.000
Covariance Type:                cluster                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          1.634

In [13]:
variables = [
    "Intercept",
    "ln_gdp_partner",
    "ln_distance",
    "contig",
    "comlang_off",
    "colony",
    "RTA"
]

comparison_ppml = pd.DataFrame({
    "OLS Coefficient": ols_model.params.reindex(variables),
    "OLS Robust SE": ols_model.bse.reindex(variables),
    "OLS P-value": ols_model.pvalues.reindex(variables),
    "PPML Coefficient": ppml_model.params.reindex(variables),
    "PPML Robust SE": ppml_model.bse.reindex(variables),
    "PPML P-value": ppml_model.pvalues.reindex(variables)
})

display(comparison_ppml.round(4))

,OLS Coefficient,OLS Robust SE,OLS P-value,PPML Coefficient,PPML Robust SE,PPML P-value
Intercept,8.1506,2.5502,0.0014,1.6344,1.4833,0.2705
ln_gdp_partner,1.0796,0.0325,0.0000,0.8436,0.0496,0.0000
ln_distance,-1.0586,0.1210,0.0000,-0.9355,0.1180,0.0000
contig,0.2477,0.3814,0.5161,0.4108,0.2204,0.0623
comlang_off,1.6689,0.3067,0.0000,1.2725,0.2281,0.0000
colony,-0.5372,0.3528,0.1278,-0.6669,0.2077,0.0013
RTA,0.2036,0.1491,0.1721,0.0694,0.1452,0.6327
